In [1]:
from IPython.display import display
import pandas as pd

from qa_model import Example, align_span, load_tokenizer, make_features

tokenizer = load_tokenizer()
question = "What is the capital of France?"
context = "The capital of France is Paris."
gold = "Paris"
first = context.index(gold)
last = first + len(gold)
example = Example("paris", question, context, ((gold, first),))

raw = tokenizer(
    [question, question], [context, "Berlin is in Germany."],
    truncation="only_second", max_length=384, stride=128,
    return_overflowing_tokens=True, return_offsets_mapping=True,
)
print("overflow_to_sample_mapping:", raw["overflow_to_sample_mapping"])
fields = {
    "input_ids": raw["input_ids"][0],
    "attention_mask": raw["attention_mask"][0],
    "sequence_id": raw.sequence_ids(0),
    "offset_mapping": raw["offset_mapping"][0],
}
if "token_type_ids" in raw:
    fields["token_type_ids"] = raw["token_type_ids"][0]
table = pd.DataFrame(fields)
table.insert(0, "token", tokenizer.convert_ids_to_tokens(fields["input_ids"]))
display(table)
features = make_features([example], tokenizer, 384, 128)
feature = features[0]
assert align_span(feature.offsets, first, last) == (feature.start, feature.end)
assert feature.start == feature.end
assert context[feature.offsets[feature.start][0]:feature.offsets[feature.end][1]] == gold
print(f"Character indices: first={first}, last inclusive={last - 1}, interval=[{first}, {last})")
print(f"Whitespace word index={len(context[:first].split())}, pair token start={feature.start}, end={feature.end}")

C:\Users\truon\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


overflow_to_sample_mapping: [0, 1]


,token,input_ids,attention_mask,sequence_id,offset_mapping,token_type_ids
0,[CLS],101,1,NaN,"(0, 0)",0
1,what,2054,1,0.0,"(0, 4)",0
2,is,2003,1,0.0,"(5, 7)",0
3,the,1996,1,0.0,"(8, 11)",0
4,capital,3007,1,0.0,"(12, 19)",0
5,of,1997,1,0.0,"(20, 22)",0
6,france,2605,1,0.0,"(23, 29)",0
7,?,1029,1,0.0,"(29, 30)",0
8,[SEP],102,1,NaN,"(0, 0)",0
9,the,1996,1,1.0,"(0, 3)",1


Character indices: first=25, last inclusive=29, interval=[25, 30)
Whitespace word index=5, pair token start=14, end=14


In [2]:
rows = []
for answer in ("Paris", "aris", "Pari", "ari", "capital of France"):
    start = context.index(answer)
    end = start + len(answer)
    token_start, token_end = align_span(feature.offsets, start, end)
    left, right = feature.offsets[token_start][0], feature.offsets[token_end][1]
    rows.append((answer, (start, end), (token_start, token_end), context[left:right]))
assert rows[1][3] == rows[2][3] == rows[3][3] == "Paris"
display(pd.DataFrame(rows, columns=["Gold", "Character range", "Token range", "Decoded span"]))

,Gold,Character range,Token range,Decoded span
0,Paris,"(25, 30)","(14, 14)",Paris
1,aris,"(26, 30)","(14, 14)",Paris
2,Pari,"(25, 29)","(14, 14)",Paris
3,ari,"(26, 29)","(14, 14)",Paris
4,capital of France,"(4, 21)","(10, 12)",capital of France


In [3]:
long_context = "alpha " * 40 + "red green blue yellow orange" + " beta" * 55
answer = "red green blue yellow orange"
start = long_context.index(answer)
end = start + len(answer)
long_example = Example("window", "Which colors?", long_context, ((answer, start),))
windows = make_features([long_example], tokenizer, max_length=18, stride=5)
rows = []
for index, window in enumerate(windows):
    offsets = [offset for offset in window.offsets if offset is not None]
    left, right = offsets[0][0], offsets[-1][1]
    complete = left <= start and end <= right
    intersects = max(left, start) < min(right, end)
    state = "complete" if complete else ("truncated" if intersects else "outside")
    cls = window.inputs["input_ids"].index(tokenizer.cls_token_id)
    assert ((window.start, window.end) != (cls, cls)) == complete
    rows.append((index, window.example_index, (left, right), state, (window.start, window.end)))
assert {row[3] for row in rows} == {"complete", "truncated", "outside"}
display(pd.DataFrame(rows, columns=["Window", "Example index", "Context range", "Answer status", "Targets"]))

,Window,Example index,Context range,Answer status,Targets
0,0,0,"(0, 71)",outside,"(0, 0)"
1,1,0,"(42, 113)",outside,"(0, 0)"
2,2,0,"(84, 155)",outside,"(0, 0)"
3,3,0,"(126, 197)",outside,"(0, 0)"
4,4,0,"(168, 239)",outside,"(0, 0)"
5,5,0,"(210, 278)",complete,"(10, 14)"
6,6,0,"(250, 313)",truncated,"(0, 0)"
7,7,0,"(289, 348)",outside,"(0, 0)"
8,8,0,"(324, 383)",outside,"(0, 0)"
9,9,0,"(359, 418)",outside,"(0, 0)"
